<a href="https://colab.research.google.com/github/lee-woonju/study-hongong-mldl/blob/main/07_2_%EC%8B%AC%EC%B8%B5_%EC%8B%A0%EA%B2%BD%EB%A7%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## KEYWORD : 심층 신경망, 렐루 함수, 옵티마이저


# 2개의 층


In [1]:
# 케라스 API를 사용해 패션 MNIST 데이터셋 불러오기
import keras
(train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()



In [2]:
from sklearn.model_selection import train_test_split

train_scaled = train_input/255.0  # 이미지 픽셀값을 0~255 범위에서 0~1 사이로 변환
train_scaled = train_scaled.reshape(-1, 28*28) # 2차원 배열을 1차원 배열로 변환
train_scaled, val_scaled, train_target, val_target = train_test_split(train_scaled, train_target, test_size=0.2, random_state=42)
# 훈련세트를 다시 훈련 세트와 검증 세트로 나눔

## 은닉층
- 입력층과 출력층 사이에 있는 모든 층

### [용어] 활성화 함수
- 신경망 층의 선형 방정식의 계산 값에 적용하는 함수
- 시그모이드, 렐루, 소프트맥스, 하이퍼볼릭 탄젠트 등이 있음

#### 시그모이드
- 뉴런의 출력 $z$값을 0과 1사이로 압축

#### 소프트맥스
- 다중 클래스 분류 문제에서 각 클래스에 속할 확률을 출력할 때 사용


In [3]:
# 시그모이드 활성화 함수를 사용한 은닉층과
# 소프트맥스 함수를 사용한 출력층을 케라스의 Dense 클래스로 만들어보자

# 입력층 Input
inputs = keras.layers.Input(shape=(784,)) # 신경망의 입력층을 정의
dense1 = keras.layers.Dense(100, activation='sigmoid')
dense2 = keras.layers.Dense(10, activation='softmax')

# 심층 신경망 만들기
- inputs, dense1, dense2 객체를 Sequential 클래스에 추가 $⇒$ **심층 신경망** 만들기


In [4]:
# 신경망 모델 만들기
model = keras.Sequential([inputs, dense1, dense2])
# keras.Sequential 케라스에서 층을 순서대로 쌓아 올리는 가장 간단한 방법

> 층을 추가하여 입력 데이터에 대해 연속적인 학습 진행 $⇒$ 인공 신경망의 강력한 성능

= 각 층이 데이터를 처리하고 변환하는 과정을 통해 모델 전체가 학습한다
  - 각 층이 학습하는 방식 : 가중치와 편향 파라미터 조정
    - ax1 + bx2 + c 라는 선형방정식이 있을 때, ab는 가중치, c는 편향
  - [헷갈렸던 부분] 각 층이 데이터의 특성을 뜻하는 줄 알았음
    - 데이터가 가진 특성은 초기 **입력층**에 주어짐
    - 각 은닉층은 이 초기 입력 특성들을 조합/변형하여 새로운, 더 추상적인 **특징**을 추출하고 생성. 다음 층의 입력으로 사용됨
    - $\therefore$ 층 = 새로운 특성들을 만들어 내는 과정이자 결과물

- [?] 그러면 은닉층에서 활성화 함수는 어떤 역할?
  - [!] **비선형성의 추가**
  - 선형 방정식은 여러 층을 쌓아도 결국 하나의 커다란 선형 방정식이 됨
  - 비선형성 부여 = 데이터 사이의 복잡한 관계(패턴)를 학습할 수 있게 함

- [?] 시그모이드 함수는 확률을 구하는 함수가 아닌지?
  - [!] 어떤 층에서 쓰이느냐에 따라 그 목적이 달라짐
  - 출력층에서는 '확률, 결과 확인용' 개념으로 사용
  - 은닉층에서는 **'특징 변환용'**으로 사용
  - 층이 깊어질 수록 기울기가 소실되어 잘 안씀
  $⇒$ **렐루**!


In [5]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

# 층을 추가하는 다른 방법


In [6]:
# 바로 클래스 객체 만들어서 층 추가
model = keras.Sequential([
    keras.layers.Input(shape=(784,)),
    keras.layers.Dense(100, activation='sigmoid', name='은닉층'),
    keras.layers.Dense(10, activation='softmax', name='출력층')
], name='패션 MNIST 모델')

model.summary()

Model: "패션 MNIST 모델"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ 은닉층 (Dense)                  │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ 출력층 (Dense)                  │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# add 메서드
model = keras.Sequential()
model.add(keras.layers.Input(shape=(784,)))
model.add(keras.layers.Dense(100, activation='sigmoid'))
model.add(keras.layers.Dense(10, activation='softmax'))

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

## 모델 훈련하기


In [10]:
model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_scaled, train_target, epochs=5)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8059 - loss: 0.5696
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8506 - loss: 0.4108
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8633 - loss: 0.3750
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8724 - loss: 0.3526
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8789 - loss: 0.3347


# 렐루 함수
- 시그모이드 함수를 양쪽 끝으로 갈수록 그래프가 누워(기울기가 소실) 올바른 출력을 만드는데 신속하게 대응 못함

- [?] 기울기 소실 = 신속 대응 못함
  - 기울기 소실 = 피드백이 거의 없음
  - 인공 신경망은 오차를 바탕으로 가중치를 수정하며 학습하기 때문에 '기울기'가 중요
  - 기울기가 크다 $→$ 많이 틀리다 $→$ 확 바꿔! $⇒$ 빠른 학습

- ReLU 함수는
  - 입력값이 양수 : 입력 통과
  - 입력값이 음수 : 0으로 만듦
  - max(0, z)
- 이미지 처리에서 좋은 성능을 냄

## Flatten 클래스

In [11]:
# Flatten 클래스는 배치 차원을 제외하고
# 나머지 입력 차원을 모두 일렬로 펼치는 역할만 함

# 배치차원 ?
# (100, 28, 28) 에서 100

# Flatten 통과하면 (100, 28, 28) => (100, 784)

model = keras.Sequential()
model.add(keras.layers.Input(shape=(28, 28))) # Flatten이 펴줄거니까 (784,)가 아니라 (28, 28)로 입력
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(100, activation='relu'))
model.add(keras.layers.Dense(10, activation='softmax'))

# Flatten은 학습하는 층이 아님
# => 깊이가 3인 신경망이라고 부르지 않음

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

- Flatten 클래스에 포함되는 모델 클래스는 0개


In [16]:
(train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()
train_scaled = train_input/255.0
train_scaled, val_scaled, train_target, val_target = train_test_split(train_scaled, train_target, test_size=0.2, random_state=42)

model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_scaled, train_target, epochs=5)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8912 - loss: 0.3078
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8955 - loss: 0.2948
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8982 - loss: 0.2858
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9008 - loss: 0.2805
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9051 - loss: 0.2705


> 크지 않지만 '시그모이드' 함수를 사용했을 때보다 성능이 조금 향상됨

In [18]:
# 검증세트에서의 성능 확인
model.evaluate(val_scaled, val_target)


375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8852 - loss: 0.3508
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9168 - loss: 0.2341


[0.23407310247421265, 0.9167500138282776]


- [?] 왜 훈련 세트 성능은 evaluate로 확인하지 않는지?
  - 이미 fit 로그에 다 나옴
  - fit 하는 동안 공부와 시험이 동시에 일어나기 때문에 굳이..
  - evaluate 하는 경우 : 과대적합 여부 판단할 때

# 옵티마이저
## 하이퍼파라미터
- 사람이 직접 입력해줘야 하는 설정값
- 모델 구조 : 은닉층의 개수, 뉴런 개수, 활성화 함수 종류 등
- 학습 전략 : **옵티마이저**의 종류, 학습률, 배치 크기
- 반복 횟수 : 에포크

## 옵티마이저
- 어떤 길잡이(알고리즘)를 쓸 것인가
  - 세부 : 학습률 설정
  - 세부 : 가속도 전략(모멘텀 최적화, 네스테로프 모멘텀 최적화)


In [19]:
# sgd : 확률적 경사 하강법
model.compile(optimizer='sgd', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 학습률 (sgd 기본 = 0.01)
sgd = keras.optimizers.SGD(learning_rate=0.1)

# 가속도 전략
## 모멘텀 최적화
## 네스테로프 모멘텀 최적화
sgd = keras.optimizers.SGD(momentum=0.9, nesterov=True)

---
# [요약]
- keras
  - keras.Sequential() : 층을 순서대로 쌓아 올리는 가장 간단한 방법, 모델 구성할 때 사용
  - Input() : 입력층 정의
  - Dense() : 밀집층 정의
- 렐루 함수
  - 시그모이드 함수의 기울기 소실을 해결
  - 층이 깊어져도 더 빠르고 성능 좋게 진행되도록 돕는 활성화 함수
  - 입력값이 양수면 그대로, 음수면 0으로 내보냄(단순)
  - 따라서, 깊은층 설계가 가능해짐(딥러닝)
  - 은닉층 activation='relu'
